## Download the crime record

In [8]:
url = 'https://files.crimestatistics.vic.gov.au/2023-06/Data_Tables_Recorded_Offences_Visualisation_Year_Ending_March_2023.xlsx'

In [9]:
# Download the dataset from the website
from urllib.request import urlretrieve
import os
output_relative_direction = '../../data/landing/'

if not os.path.exists(output_relative_direction):
    os.makedirs(output_relative_direction)
    
target_direction = 'external_data'
if not os.path.exists(output_relative_direction  + target_direction):
    os.makedirs(output_relative_direction + target_direction)

url="https://files.crimestatistics.vic.gov.au/2023-06/Data_Tables_LGA_Recorded_Offences_Year_Ending_March_2023.xlsx"
name_file = 'Crime'
output_direction = output_relative_direction + target_direction
output_direction = f"{output_direction}/{name_file}.csv"

urlretrieve(url, output_direction) 

('../../data/landing/external_data/Crime.csv',
 <http.client.HTTPMessage at 0x7fee6afcf6a0>)

## Preprocess the crime

In [10]:
import pandas as pd

In [11]:
# Read the file, and list the sheet name of the excel file 
crime_file = pd.ExcelFile('../../data/landing/external_data/Crime.csv')
sheet_list=crime_file.sheet_names
sheet_list

['Contents',
 'Footnotes',
 'Table 01',
 'Table 02',
 'Table 03',
 'Table 04',
 'Table 05',
 'Table 06']

In [17]:
# Read the crime csv file, and choose the sheet with crime count in each postcode
read_crime=pd.read_excel('../../data/landing/external_data/Crime.csv',sheet_name = ['Table 03'])['Table 03']
#read_crime = read_crime.dropna()
read_crime.to_csv('../../data/raw/external_data/temp_crime.csv')
read_crime.count()


Year                     359132
Year ending              359132
Local Government Area    359132
Postcode                 359132
Suburb/Town Name         359132
Offence Division         359132
Offence Subdivision      359132
Offence Subgroup         359132
Offence Count            359132
dtype: int64

In [18]:
# Keep the crime count in 2021, 2022, 2023
read_crime  = read_crime[read_crime['Year'].isin([2023,2022,2021])]
read_crime.to_csv('../../data/raw/external_data/temp_crime.csv')
# Drop the feature which we don't need 
curated_crime=read_crime.drop(columns=['Offence Division','Offence Subdivision','Offence Subgroup'])

In [21]:
curated_crime

,Year,Year ending,Local Government Area,Postcode,Suburb/Town Name,Offence Count
0,2023,March,Alpine,3691,Dederang,1
1,2023,March,Alpine,3691,Dederang,1
2,2023,March,Alpine,3691,Dederang,2
3,2023,March,Alpine,3691,Dederang,1
4,2023,March,Alpine,3691,Dederang,1
...,...,...,...,...,...,...
111061,2021,March,Yarriambiack,3491,Patchewollock,4
111062,2021,March,Yarriambiack,3491,Patchewollock,2
111063,2021,March,Yarriambiack,3491,Patchewollock,1
111064,2021,March,Yarriambiack,3491,Patchewollock,2


In [22]:
# Group by the offence count by postcode and divided by 3 to find the offence count each year in each postcode
count_crimes= curated_crime.groupby('Postcode')['Offence Count'].count().reset_index()
count_crimes['Offence Count'] = count_crimes['Offence Count']/3

In [23]:
# Rename the offence count into crime_count, Postcode into postcode
count_crimes = count_crimes.rename(columns={'Offence Count': 'crime_count'})
count_crimes = count_crimes.rename(columns={'Postcode': 'postcode'})
count_crimes
count_crimes.to_csv('../../data/curated/external_data/Crime.csv', index=False)
count_crimes.head(10)

,postcode,crime_count
0,3000,86.666667
1,3002,55.666667
2,3003,54.666667
3,3004,118.666667
4,3006,117.333333
5,3008,65.333333
6,3011,96.666667
7,3012,206.000000
8,3013,49.333333
9,3015,102.000000
